# eBird API
## MAD 2502 — Applied Data

This tutorial introduces a real public-facing data service: the **eBird API**.

You will practice the full path

**API key → request → JSON → DataFrame → question → result**

The goal is not to memorize one URL. The goal is to understand the pattern well enough to recognize it in another API.


## 1. Keep your API key private

Create a file named `my_api_info.py` in your MAD2502 folder:

```python
ebird_api_key = "PASTE-YOUR-KEY-HERE"
```

That file is listed in this repository's `.gitignore`, so Git should not upload it.

**Never put a real API key directly into a public notebook.**


In [ ]:
import pandas as pd
import requests

try:
    import my_api_info
    API_KEY = my_api_info.ebird_api_key
    print("API key loaded.")
except (ImportError, AttributeError):
    API_KEY = None
    print("No API key found. The offline practice examples will still work.")


## 2. Anatomy of a request

We will use eBird's recent-observations endpoint:

```text
https://api.ebird.org/v2/data/obs/{region_code}/recent/{species_code}
```

Examples:

- Florida: `US-FL`
- Northern Cardinal: `norcar`
- Blue Jay: `blujay`
- American Crow: `amecro`

The API key goes in a request **header**. Options such as the number of recent days go in `params`.


In [ ]:
region_code = "US-FL"
species_code = "norcar"
days = 7

url = (
    "https://api.ebird.org/v2/data/obs/"
    f"{region_code}/recent/{species_code}"
)

headers = {"X-eBirdApiToken": API_KEY}
params = {"back": days, "maxResults": 5000}

print(url)
print(params)


## 3. Make the request

Only run the live request if your key loaded successfully.

`raise_for_status()` is important: a failed web request should produce an error instead of silently pretending that empty or incorrect data are valid.


In [ ]:
if API_KEY:
    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30,
    )
    response.raise_for_status()
    records = response.json()
    print("records returned:", len(records))
else:
    print("Skipping live request: add your key to my_api_info.py first.")


## 4. What does JSON become in Python?

A common API response is a **list of dictionaries**. That maps naturally into a pandas DataFrame.

To practice without a key or internet connection, use these pretend records first.


In [ ]:
practice_records = [
    {
        "speciesCode": "norcar",
        "comName": "Northern Cardinal",
        "locName": "Campus Garden",
        "obsDt": "2026-10-20 08:10",
        "howMany": 2,
    },
    {
        "speciesCode": "norcar",
        "comName": "Northern Cardinal",
        "locName": "Lake Trail",
        "obsDt": "2026-10-21 09:25",
        "howMany": 5,
    },
    {
        "speciesCode": "norcar",
        "comName": "Northern Cardinal",
        "locName": "Wetlands",
        "obsDt": "2026-10-22 07:55",
        "howMany": None,
    },
]

practice_birds = pd.DataFrame(practice_records)
practice_birds


### Important distinction: rows vs. birds

`len(df)` counts **returned records**.

`df["howMany"].sum()` adds the known bird counts.

Those are different questions. One observation row reporting 12 birds is still one row.


In [ ]:
counts = pd.to_numeric(practice_birds["howMany"], errors="coerce")

print("returned records:", len(practice_birds))
print("known bird count total:", counts.sum())
print("missing counts:", counts.isna().sum())


A missing `howMany` is **unknown**, not automatically zero and not automatically one. How you handle missing values depends on the question you are asking.

For population questions, remember that this endpoint is not a census of every bird in Florida.


## 5. Turn the live response into a DataFrame


In [ ]:
if API_KEY:
    birds = pd.DataFrame(records)

    useful_columns = [
        "speciesCode",
        "comName",
        "locId",
        "locName",
        "obsDt",
        "howMany",
    ]

    available = [name for name in useful_columns if name in birds.columns]
    birds = birds[available].copy()

    if "howMany" in birds:
        birds["howMany"] = pd.to_numeric(
            birds["howMany"],
            errors="coerce",
        )

    display(birds.head())
    print("shape:", birds.shape)


## 6. Ask several kinds of questions

With a live `birds` DataFrame, try:

```python
len(birds)
birds["howMany"].sum()
birds["howMany"].mean()
birds["locName"].nunique()
birds["locName"].value_counts().head(10)
```

Notice that each line answers a different question.


In [ ]:
if API_KEY:
    print("records:", len(birds))
    print("known count total:", birds["howMany"].sum())
    print("mean known count:", birds["howMany"].mean())
    print("unique locations:", birds["locName"].nunique())


## 7. Write a reusable function

Functions make your code easier to reuse for another species or region.


In [ ]:
def get_ebird_data(
    api_key,
    region_code,
    species_code,
    days=7,
    max_results=5000,
):
    """Return recent eBird records for one species and region."""
    endpoint = (
        "https://api.ebird.org/v2/data/obs/"
        f"{region_code}/recent/{species_code}"
    )
    request_headers = {"X-eBirdApiToken": api_key}
    request_params = {
        "back": days,
        "maxResults": max_results,
    }

    response = requests.get(
        endpoint,
        headers=request_headers,
        params=request_params,
        timeout=30,
    )
    response.raise_for_status()

    data = pd.DataFrame(response.json())

    if "howMany" in data:
        data["howMany"] = pd.to_numeric(
            data["howMany"],
            errors="coerce",
        )

    return data


In [ ]:
if API_KEY:
    blue_jays = get_ebird_data(
        API_KEY,
        "US-FL",
        "blujay",
        days=10,
    )

    print("Blue Jay records:", len(blue_jays))
    display(blue_jays.head())


# Practice

1. Retrieve Northern Cardinal (`norcar`) data for Florida for the last 7 days.
2. Count the number of returned records.
3. Add the known `howMany` values.
4. Change the species to Blue Jay (`blujay`).
5. Change the region to another state.
6. Find the five locations appearing most often in the returned data.
7. Explain in one sentence why a sum of `howMany` is **not** a population estimate.

### Capstone connection

When evaluating a new API, ask:

- What exactly does one returned record represent?
- Are there limits or caps?
- Can values be missing?
- Does the service require authentication?
- Are there usage rules?
- Can you reproduce the request later?
